In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
import optuna
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split,RandomizedSearchCV,GridSearchCV
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

/Users/kalp/PycharmProjects/Machine_learning/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('Travel.csv')
df

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,3,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,1.0,Manager,26576.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,4,5.0,Basic,3.0,Single,3.0,1,3,1,2.0,Executive,21212.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4,4.0,Standard,4.0,Married,7.0,0,1,1,3.0,Senior Manager,31820.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,3,4.0,Basic,3.0,Single,3.0,0,5,0,2.0,Executive,20289.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                4888 non-null   int64  
 1   ProdTaken                 4888 non-null   int64  
 2   Age                       4662 non-null   float64
 3   TypeofContact             4863 non-null   object 
 4   CityTier                  4888 non-null   int64  
 5   DurationOfPitch           4637 non-null   float64
 6   Occupation                4888 non-null   object 
 7   Gender                    4888 non-null   object 
 8   NumberOfPersonVisiting    4888 non-null   int64  
 9   NumberOfFollowups         4843 non-null   float64
 10  ProductPitched            4888 non-null   object 
 11  PreferredPropertyStar     4862 non-null   float64
 12  MaritalStatus             4888 non-null   object 
 13  NumberOfTrips             4748 non-null   float64
 14  Passport

In [5]:
df.isnull().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

# DATA CLEANING

In [6]:
df.columns

Index(['CustomerID', 'ProdTaken', 'Age', 'TypeofContact', 'CityTier',
       'DurationOfPitch', 'Occupation', 'Gender', 'NumberOfPersonVisiting',
       'NumberOfFollowups', 'ProductPitched', 'PreferredPropertyStar',
       'MaritalStatus', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore',
       'OwnCar', 'NumberOfChildrenVisiting', 'Designation', 'MonthlyIncome'],
      dtype='object')

In [7]:
cols = ['TypeofContact', 'Occupation','Gender', 'ProductPitched','MaritalStatus','Designation']
for i in cols:
    print(df[i].value_counts())
    '\n'

TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64


In [8]:
df['Gender'] = df['Gender'].replace('Fe Male', 'Female')
df['MaritalStatus'] = df['MaritalStatus'].replace('Single','Unmarried')

In [9]:
df['Gender'].value_counts()

Gender
Male      2916
Female    1972
Name: count, dtype: int64

In [10]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married      2340
Unmarried    1598
Divorced      950
Name: count, dtype: int64

In [11]:
features_with_nan = [features for features in df.columns if df[features].isnull().sum()>=1 ]
for i in features_with_nan:
    print(i,np.round(df[i].isnull().mean()*100,5),'%')

Age 4.62357 %
TypeofContact 0.51146 %
DurationOfPitch 5.13502 %
NumberOfFollowups 0.92062 %
PreferredPropertyStar 0.53191 %
NumberOfTrips 2.86416 %
NumberOfChildrenVisiting 1.35025 %
MonthlyIncome 4.76678 %


In [12]:
df[features_with_nan].select_dtypes(exclude='object').describe()

,Age,DurationOfPitch,NumberOfFollowups,PreferredPropertyStar,NumberOfTrips,NumberOfChildrenVisiting,MonthlyIncome
count,4662.000000,4637.000000,4843.000000,4862.000000,4748.000000,4822.000000,4655.000000
mean,37.622265,15.490835,3.708445,3.581037,3.236521,1.187267,23619.853491
std,9.316387,8.519643,1.002509,0.798009,1.849019,0.857861,5380.698361
min,18.000000,5.000000,1.000000,3.000000,1.000000,0.000000,1000.000000
25%,31.000000,9.000000,3.000000,3.000000,2.000000,1.000000,20346.000000
50%,36.000000,13.000000,4.000000,3.000000,3.000000,1.000000,22347.000000
75%,44.000000,20.000000,4.000000,4.000000,4.000000,2.000000,25571.000000
max,61.000000,127.000000,6.000000,5.000000,22.000000,3.000000,98678.000000


In [13]:
df['Age'].fillna(df['Age'].median(),inplace=True)
df['TypeofContact'].fillna(df['TypeofContact'].mode()[0],inplace=True)
df['DurationOfPitch'].fillna(df['DurationOfPitch'].median(),inplace=True)
df['NumberOfFollowups'].fillna(df['NumberOfFollowups'].median(),inplace=True)
df['PreferredPropertyStar'].fillna(df['PreferredPropertyStar'].mode()[0],inplace=True)
df['NumberOfTrips'].fillna(df['NumberOfTrips'].median(),inplace=True)
df['NumberOfChildrenVisiting'].fillna(df['NumberOfChildrenVisiting'].mode()[0],inplace=True)
df['MonthlyIncome'].fillna(df['MonthlyIncome'].median(),inplace=True)

In [14]:
df.isnull().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [15]:
df.drop(columns=['CustomerID'],axis=1,inplace=True)

In [16]:
df.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0
1,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0
3,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,0,36.0,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [17]:
df['TotalVisiting'] = df['NumberOfPersonVisiting'] + df['NumberOfChildrenVisiting']
df.drop(columns=['NumberOfChildrenVisiting','NumberOfPersonVisiting'],axis=1,inplace=True)

In [18]:
num_features = [feature for feature in df.columns if df[feature].dtypes!='object']
print(num_features)

['ProdTaken', 'Age', 'CityTier', 'DurationOfPitch', 'NumberOfFollowups', 'PreferredPropertyStar', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar', 'MonthlyIncome', 'TotalVisiting']


In [19]:
cat_features = [feature for feature in df.columns if df[feature].dtypes=='object']
print(cat_features)

['TypeofContact', 'Occupation', 'Gender', 'ProductPitched', 'MaritalStatus', 'Designation']


In [20]:
discrete_features = [feature for feature in num_features if len(df[feature].unique())<=25]
print(discrete_features)

['ProdTaken', 'CityTier', 'NumberOfFollowups', 'PreferredPropertyStar', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar', 'TotalVisiting']


In [21]:
cont_features = [feature for feature in num_features if feature not in discrete_features]
print(cont_features)

['Age', 'DurationOfPitch', 'MonthlyIncome']


In [22]:
X = df.drop(columns=['ProdTaken'],axis=1)
y = df['ProdTaken']

In [23]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [24]:
y.value_counts()

ProdTaken
0    3968
1     920
Name: count, dtype: int64

## COLUMN TRANSFORMER

This estimator allows different columns or column subsets of the input to be transformed separately and the features generated by each transformer will be concatenated to form a single feature space. This is useful for heterogeneous or columnar data, to combine several feature extraction mechanisms or transformations into a single transformer.

In [25]:
cate_features = X.select_dtypes(include='object').columns
nume_features = X.select_dtypes(exclude='object').columns

ohe = OneHotEncoder(drop='first')
numeric_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    [
        ('ohe', ohe, cate_features),
        ('numeric', numeric_transformer, nume_features)
    ]
)

In [26]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)
X_train

array([[ 1.        ,  0.        ,  0.        , ...,  0.78296635,
        -0.38224537, -0.77415132],
       [ 1.        ,  0.        ,  1.        , ...,  0.78296635,
        -0.4597992 ,  0.64361526],
       [ 1.        ,  1.        ,  0.        , ...,  0.78296635,
        -0.24519557, -0.06526803],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.78296635,
        -0.36057591,  0.64361526],
       [ 1.        ,  0.        ,  0.        , ...,  0.78296635,
        -0.25279888,  0.64361526],
       [ 0.        ,  0.        ,  1.        , ..., -1.2771941 ,
        -1.08251091, -1.48303461]])

In [27]:
models = {
    'Random Forest':RandomForestClassifier(),
    'AdaBoost':AdaBoostClassifier(),
    'Decision Tree':DecisionTreeClassifier(),
    'Logistic Regression':LogisticRegression(),
    'Xgboost':XGBClassifier()
}
for i in range(len(list(models))):
    model = models[list(models)[i]]
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    print(f'{list(models)[i]} Accuracy: {accuracy_score(y_test,y_pred)}')
    print(f'{list(models)[i]} Confusion Matrix: \n{confusion_matrix(y_test,y_pred)}')
    print(f'{list(models)[i]} Classification Report: \n{classification_report(y_test,y_pred)}')
    print('\n')

Random Forest Accuracy: 0.9314928425357873
Random Forest Confusion Matrix: 
[[784   3]
 [ 64 127]]
Random Forest Classification Report: 
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       787
           1       0.98      0.66      0.79       191

    accuracy                           0.93       978
   macro avg       0.95      0.83      0.88       978
weighted avg       0.93      0.93      0.93       978



AdaBoost Accuracy: 0.8353783231083844
AdaBoost Confusion Matrix: 
[[772  15]
 [146  45]]
AdaBoost Classification Report: 
              precision    recall  f1-score   support

           0       0.84      0.98      0.91       787
           1       0.75      0.24      0.36       191

    accuracy                           0.84       978
   macro avg       0.80      0.61      0.63       978
weighted avg       0.82      0.84      0.80       978



Decision Tree Accuracy: 0.9192229038854806
Decision Tree Confusion Matrix: 
[[756  

# HYPER-PARAMETER TUNING

In [28]:
rf_params = {
    'max_depth': [5,8,15,None,10],
    'max_features': ['auto','sqrt','log2',5,6,7],
    'n_estimators': [100,500,1000],
    'min_samples_split': [2,5,10,15,20]
}
ada_params = {
    'n_estimators': [10,50,100,250,500],
    'learning_rate': [0.001,0.01,0.1,1,10],
}

In [29]:
cv_models = [
    ('RF',RandomForestClassifier(),rf_params),
    ('AdaBoost',AdaBoostClassifier(),ada_params)
]

In [30]:
model_params = {}

for name,model,params in cv_models:
    random = RandomizedSearchCV(estimator=model,param_distributions=params,n_iter=10,cv=3,verbose=2,n_jobs=-1)
    random.fit(X_train,y_train)
    model_params[name] = random.best_params_
    print(f'Best Parameters for {name}: {model_params[name]}')
    grid = GridSearchCV(estimator=model,param_grid=params,cv=3,verbose=2,n_jobs=-1)
    grid.fit(X_train,y_train)
    print(f'Best Parameters for {name}: {grid.best_params_}')
    print('\n')

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END max_depth=None, max_features=6, min_samples_split=20, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=6, min_samples_split=20, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, max_features=6, min_samples_split=20, n_estimators=100; total time=   0.2s
[CV] END max_depth=8, max_features=6, min_samples_split=10, n_estimators=100; total time=   0.2s
[CV] END max_depth=8, max_features=6, min_samples_split=10, n_estimators=100; total time=   0.2s
[CV] END max_depth=5, max_features=sqrt, min_samples_split=2, n_estimators=500; total time=   0.8s
[CV] END max_depth=15, max_features=auto, min_samples_split=20, n_estimators=100; total time=   0.0s
[CV] END max_depth=15, max_features=auto, min_samples_split=20, n_estimators=100; total time=   0.0s
[CV] END max_depth=15, max_features=auto, min_samples_split=20, n_estimators=100; total time=   0.0s
[CV] END max_depth=5, max_features=sqrt, mi

KeyboardInterrupt: 

In [35]:
grid.best_params_

{'learning_rate': 1, 'n_estimators': 100}

In [36]:
random.best_params_

{'n_estimators': 500, 'learning_rate': 1}

In [37]:
y_pred = random.predict(X_test)
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

0.8394683026584867
              precision    recall  f1-score   support

           0       0.85      0.97      0.91       787
           1       0.71      0.30      0.42       191

    accuracy                           0.84       978
   macro avg       0.78      0.64      0.67       978
weighted avg       0.82      0.84      0.81       978

[[763  24]
 [133  58]]


In [38]:
y_pred = grid.predict(X_test)
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

0.8384458077709611
              precision    recall  f1-score   support

           0       0.84      0.98      0.91       787
           1       0.79      0.24      0.36       191

    accuracy                           0.84       978
   macro avg       0.82      0.61      0.64       978
weighted avg       0.83      0.84      0.80       978

[[775  12]
 [146  45]]


In [39]:
def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate',0,1),
        'n_estimators': trial.suggest_int('n_estimators',low=5,high=1000,log=True)
    }
    model = AdaBoostClassifier(**params)
    score = accuracy_score(y_test,y_pred)
    return score
study = optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=1000)
print(study.best_params)

[I 2025-11-01 14:50:37,672] A new study created in memory with name: no-name-48ecbd42-5871-405e-9add-98507511c394
[I 2025-11-01 14:50:37,680] Trial 0 finished with value: 0.8384458077709611 and parameters: {'learning_rate': 0.9766218470915263, 'n_estimators': 54}. Best is trial 0 with value: 0.8384458077709611.
[I 2025-11-01 14:50:37,682] Trial 1 finished with value: 0.8384458077709611 and parameters: {'learning_rate': 0.40323593006052727, 'n_estimators': 36}. Best is trial 0 with value: 0.8384458077709611.
[I 2025-11-01 14:50:37,683] Trial 2 finished with value: 0.8384458077709611 and parameters: {'learning_rate': 0.5575767982969759, 'n_estimators': 6}. Best is trial 0 with value: 0.8384458077709611.
[I 2025-11-01 14:50:37,684] Trial 3 finished with value: 0.8384458077709611 and parameters: {'learning_rate': 0.36761747898717256, 'n_estimators': 403}. Best is trial 0 with value: 0.8384458077709611.
[I 2025-11-01 14:50:37,685] Trial 4 finished with value: 0.8384458077709611 and paramete

{'learning_rate': 0.9766218470915263, 'n_estimators': 54}


In [40]:
best_params = study.best_params
boost = AdaBoostClassifier(**best_params)
boost.fit(X_train,y_train)
y_pred = boost.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.8404907975460123


## HYPER-PARAMETER TUNING XGBOOST USING OPTUNA

In [48]:
def objective(trial):
    booster = trial.suggest_categorical('booster',['gbtree','dart','gblinear'])
    params = {
        "booster":booster,
        'n_estimators':trial.suggest_int('n_estimators',low=5,high=1000,log=True),
        'learning_rate':trial.suggest_float('learning_rate',low=0.1,high=1,log=True),
    }
    if booster in ['gbtree','dart']:
        params.update({
            'max_depth':trial.suggest_int('max_depth',low=1,high=15,step=1),
            'min_child_weight':trial.suggest_int('min_child_weight',low=1,high=10,step=1),
            'gamma':trial.suggest_float('gamma',low=0.1,high=1,log=True),
            'reg_alpha':trial.suggest_float('reg_alpha',low=0.1,high=1,log=True),
            'reg_lambda':trial.suggest_float('reg_lambda',low=0.1,high=1,log=True),
        })
    model = XGBClassifier(**params)
    accuracy = accuracy_score(y_test,y_pred)
    return accuracy

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=1000)

[I 2025-11-01 14:58:02,171] A new study created in memory with name: no-name-837d27e4-ec03-4db8-86ff-4698b4cf1caa
[I 2025-11-01 14:58:02,175] Trial 0 finished with value: 0.9212678936605317 and parameters: {'booster': 'gblinear', 'n_estimators': 632, 'learning_rate': 0.29019543782476226}. Best is trial 0 with value: 0.9212678936605317.
[I 2025-11-01 14:58:02,176] Trial 1 finished with value: 0.9212678936605317 and parameters: {'booster': 'gblinear', 'n_estimators': 231, 'learning_rate': 0.3024451621079473}. Best is trial 0 with value: 0.9212678936605317.
[I 2025-11-01 14:58:02,178] Trial 2 finished with value: 0.9212678936605317 and parameters: {'booster': 'gbtree', 'n_estimators': 157, 'learning_rate': 0.7396935686641946, 'max_depth': 8, 'min_child_weight': 4, 'gamma': 0.6448756239622311, 'reg_alpha': 0.11531605929219545, 'reg_lambda': 0.13092218419377782}. Best is trial 0 with value: 0.9212678936605317.
[I 2025-11-01 14:58:02,179] Trial 3 finished with value: 0.9212678936605317 and p

In [49]:
study.best_params

{'booster': 'gblinear',
 'n_estimators': 632,
 'learning_rate': 0.29019543782476226}

In [50]:
study_params = study.best_params
xgboost = XGBClassifier(**study_params)
xgboost.fit(X_train,y_train)
y_pred = xgboost.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.83640081799591


In [1]:
def object(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth',low=1,high=15,step=1),
        'n_estimators': trial.suggest_int('n_estimators',low=5,high=1000,log=True),
        'min_sample_split': trial.suggest_int('min_sample_split',low=2,high=10,step=1)
    }
    model = RandomForestClassifier(**params)
    accuracy = accuracy_score(y_test,y_pred)
    return accuracy
study = optuna.create_study(direction='maximize')
study.optimize(object, n_trials=1000)

NameError: name 'optuna' is not defined